# Day 32: LLMOps
**MLOps vs LLMOps, fine-tuning, versioning, monitoring, and cost and latency**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas matplotlib

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas matplotlib
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Preparing fine-tuning data

In [ ]:
import json, re, random, hashlib, numpy as np, pandas as pd
import genai_helpers
random.seed(32)
templates = {
    "billing": ["I was charged twice for order {o}", "Refund for {o} not received", "Invoice amount is wrong, contact me at {e}"],
    "technical": ["App crashes when I open {o}", "Login fails with error 500", "Page is very slow, call me on {p}"],
    "delivery": ["Where is my order {o}?", "Parcel {o} delivered to the wrong address", "Delivery is 3 days late"],
}
rows = []
for label, ts in templates.items():
    for _ in range(20):
        t = random.choice(ts).format(o=f"ORD-{random.randint(1000, 9999)}", e="mei@example.com", p="+91 98450 11111")
        rows.append({"text": t, "label": label})

PII = [r"[\w.+-]+@[\w-]+\.[\w.]+", r"\+?\d{1,3}[ -]?\d{4,5}[ -]?\d{4,6}"]
def scrub(t):
    for p in PII:
        t = re.sub(p, "[redacted]", t)
    return t

SYSTEM = "Classify the support ticket as billing, technical or delivery. Reply with one word."
examples, pii_removed = [], 0
for r in rows:
    clean = scrub(r["text"]); pii_removed += clean != r["text"]
    examples.append({"messages": [{"role": "system", "content": SYSTEM},
                                  {"role": "user", "content": clean},
                                  {"role": "assistant", "content": r["label"]}]})

def valid(ex):
    m = ex["messages"]
    return ([x["role"] for x in m] == ["system", "user", "assistant"] and all(x["content"].strip() for x in m)
            and m[2]["content"] in templates)

random.shuffle(examples)
split = int(len(examples) * 0.8)
for name, part in [("train.jsonl", examples[:split]), ("val.jsonl", examples[split:])]:
    with open(name, "w") as f:
        for ex in part:
            f.write(json.dumps(ex) + "\n")
print(f"Valid: {sum(map(valid, examples))}/{len(examples)} | PII removed: {pii_removed} | train {split}, validation {len(examples) - split}")
print(examples[0])

In [ ]:
RUN_FINE_TUNE = False     # costs money: set True only with approval, an API key and a model that supports fine-tuning
if ONLINE and RUN_FINE_TUNE:
    c = genai_helpers.client
    train_file = c.files.create(file=open("train.jsonl", "rb"), purpose="fine-tune")
    val_file = c.files.create(file=open("val.jsonl", "rb"), purpose="fine-tune")
    job = c.fine_tuning.jobs.create(training_file=train_file.id, validation_file=val_file.id,
                                    model=genai_helpers.os.getenv("FT_BASE_MODEL", "gpt-4o-mini-2024-07-18"))
    print("Started fine-tuning job:", job.id)
else:
    print("Fine-tuning not started (RUN_FINE_TUNE is False). The data files are ready: train.jsonl, val.jsonl")

## 3. A prompt registry and release manifest

In [ ]:
import difflib
registry = {}
def register(pid, text, note):
    sha = hashlib.sha256(text.encode()).hexdigest()[:6]
    registry[pid] = {"text": text, "sha": sha, "note": note}
    return sha

register("answer_v7", "Answer using ONLY the context.\nCite chunk ids.\nIf unsure, say I don't know.", "baseline")
register("answer_v8", "Answer using ONLY the context.\nCite chunk ids.\nAnswer in at most 3 sentences.\nIf unsure, say I don't know.", "shorter answers to cut output tokens")
for line in difflib.unified_diff(registry["answer_v7"]["text"].splitlines(), registry["answer_v8"]["text"].splitlines(), "answer_v7", "answer_v8", lineterm=""):
    print(line)

manifest = {"release": "support-bot-1.8.0",
            "prompt": {"id": "answer_v8", "sha": registry["answer_v8"]["sha"]},
            "model": {"name": genai_helpers.MODEL, "temperature": 0},
            "retrieval": {"embed": genai_helpers.EMBED_MODEL, "chunk_tokens": 400, "k": 3},
            "golden_set": "golden_v4", "eval_report": "report_1.8.0.json"}
json.dump(manifest, open("release_manifest.json", "w"), indent=2)
print(json.dumps(manifest, indent=2))

## 4. Cost and latency from request logs
**The prices below are illustrative placeholders.** Replace them with current prices from your provider's pricing page.

In [ ]:
PRICES = {"large-model": {"in": 2.50, "out": 10.00}, "small-model": {"in": 0.15, "out": 0.60}}   # EXAMPLE USD per 1M tokens

rng = np.random.default_rng(32)
n = 6000
feature = rng.choice(["faq_bot", "summariser", "agent_research"], n, p=[0.7, 0.21, 0.09])
model = np.where(feature == "faq_bot", rng.choice(["large-model", "small-model"], n, p=[0.5, 0.5]), "large-model")
tok_in = np.where(feature == "agent_research", rng.normal(14000, 3000, n), np.where(feature == "summariser", rng.normal(3000, 600, n), rng.normal(800, 150, n))).clip(100)
tok_out = np.where(feature == "summariser", rng.normal(400, 80, n), rng.normal(180, 50, n)).clip(10)
lat = (np.where(model == "large-model", 900, 300) + tok_out * np.where(model == "large-model", 12, 4) + rng.gamma(2, 150, n))
logs = pd.DataFrame({"day": rng.choice(pd.date_range("2026-12-01", periods=7).date, n), "feature": feature, "model": model,
                     "tok_in": tok_in.round(), "tok_out": tok_out.round(), "latency_ms": lat.round()})
logs["cost"] = [(r.tok_in * PRICES[r.model]["in"] + r.tok_out * PRICES[r.model]["out"]) / 1e6 for r in logs.itertuples()]

by_feature = logs.groupby("feature").agg(requests=("cost", "size"), cost=("cost", "sum"))
by_feature["share_of_requests"] = (by_feature.requests / n).round(2)
by_feature["share_of_cost"] = (by_feature.cost / by_feature.cost.sum()).round(2)
print(by_feature.round(2).to_string(), "\n")
print("Latency by model (ms):\n", logs.groupby("model").latency_ms.quantile([0.5, 0.95]).unstack().round().to_string())

## 5. Caching repeated questions

In [ ]:
questions = [f"faq question {i}" for i in range(3000)]
weights = 1 / np.arange(1, 3001) ** 1.1; weights /= weights.sum()          # a few questions are very common
stream = rng.choice(questions, 5000, p=weights)
cache, hits = {}, 0
for q in stream:
    key = q.strip().lower()
    if key in cache:
        hits += 1
    else:
        cache[key] = f"answer to {q}"
cost_each = (800 * PRICES["large-model"]["in"] + 180 * PRICES["large-model"]["out"]) / 1e6
print(f"Hit rate {hits / len(stream):.0%} | model calls {len(stream) - hits} instead of {len(stream)} | saved {hits * cost_each:.2f} USD (example prices)")

## 6. Routing: cost vs quality
Quality numbers here are **simulated** to show the trade-off. Measure real quality on your golden set.

In [ ]:
qs = pd.DataFrame({"hard": rng.random(2000) < 0.3})
QUALITY = {("large-model", True): 0.90, ("large-model", False): 0.93, ("small-model", True): 0.70, ("small-model", False): 0.92}  # simulated
def run(policy):
    m = qs.hard.map(lambda h: policy(h))
    quality = np.mean([QUALITY[(mm, h)] for mm, h in zip(m, qs.hard)])
    cost = sum((800 * PRICES[mm]["in"] + 180 * PRICES[mm]["out"]) / 1e6 for mm in m)
    return quality, cost
base_q, base_c = run(lambda h: "large-model")
for name, pol in [("all large", lambda h: "large-model"), ("all small", lambda h: "small-model"),
                  ("routed", lambda h: "large-model" if h else "small-model")]:
    q, c = run(pol)
    print(f"{name:<10} quality {q:.3f} | cost {c / base_c:.0%} of all-large")

### TODO 1: Write a simple difficulty classifier (length, number of questions, words like "compare" or "why") and use it instead of the `hard` column.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)